Kaarina Mattika, CS87A, Section 1758
https://github.com/kmattika/cs82a-portfolio

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

In [2]:
df = pd.read_csv(r'C:\Users\kamat\OneDrive\Desktop\CS82A\custdata.tsv', sep='\t')
display(df.head(10))
display(df.shape)
display(df.info())
display(df.describe())

,custid,sex,is.employed,income,marital.stat,health.ins,housing.type,recent.move,num.vehicles,age,state.of.res
0,2068,F,NaN,11300,Married,True,Homeowner free and clear,False,2.0,49.0,Michigan
1,2073,F,NaN,0,Married,True,Rented,True,3.0,40.0,Florida
2,2848,M,True,4500,Never Married,False,Rented,True,3.0,22.0,Georgia
3,5641,M,True,20000,Never Married,False,Occupied with no rent,False,0.0,22.0,New Mexico
4,6369,F,True,12000,Never Married,True,Rented,True,1.0,31.0,Florida
5,8322,F,True,180000,Never Married,True,Homeowner with mortgage/loan,False,1.0,40.0,New York
6,8521,M,True,120000,Never Married,True,Homeowner free and clear,True,1.0,39.0,Idaho
7,12195,M,True,40000,Married,True,Rented,False,3.0,48.0,Michigan
8,14989,M,NaN,9400,Married,True,Rented,False,2.0,44.0,Illinois
9,15917,F,True,24000,Divorced/Separated,True,Homeowner free and clear,False,1.0,70.0,North Carolina


(1000, 11)

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   custid        1000 non-null   int64  
 1   sex           1000 non-null   str    
 2   is.employed   672 non-null    object 
 3   income        1000 non-null   int64  
 4   marital.stat  1000 non-null   str    
 5   health.ins    1000 non-null   bool   
 6   housing.type  944 non-null    str    
 7   recent.move   944 non-null    object 
 8   num.vehicles  944 non-null    float64
 9   age           1000 non-null   float64
 10  state.of.res  1000 non-null   str    
dtypes: bool(1), float64(2), int64(2), object(2), str(4)
memory usage: 115.8+ KB


None

,custid,income,num.vehicles,age
count,1.000000e+03,1000.000000,944.000000,1000.000000
mean,6.984997e+05,53504.771000,1.916314,51.699815
std,4.135083e+05,65478.065729,1.101618,18.863433
min,2.068000e+03,-8700.000000,0.000000,0.000000
25%,3.456668e+05,14600.000000,1.000000,38.000000
50%,6.934030e+05,35000.000000,2.000000,50.000000
75%,1.044606e+06,67000.000000,2.000000,64.000000
max,1.414286e+06,615000.000000,6.000000,146.680197


In [3]:
df[['age', 'income', 'num.vehicles']].corr()

,age,income,num.vehicles
age,1.000000,0.027429,-0.063666
income,0.027429,1.000000,0.139198
num.vehicles,-0.063666,0.139198,1.000000


In [4]:
df.dropna(
    subset=['income', 'num.vehicles', 'age'],
    inplace=True
)

display(df.shape)

(944, 11)

In [5]:
df[
    (df['income'] < 0) |
    (df['age'] >= 100) |
    (df['age'] <= 18)
]

,custid,sex,is.employed,income,marital.stat,health.ins,housing.type,recent.move,num.vehicles,age,state.of.res
212,287882,M,True,60000,Married,True,Rented,True,2.0,137.700030,Florida
266,364991,F,NaN,12000,Divorced/Separated,True,Homeowner with mortgage/loan,False,1.0,123.061023,Nevada
286,397230,F,True,31200,Married,True,Rented,False,2.0,136.052160,Texas
593,843811,F,True,40000,Never Married,True,Homeowner with mortgage/loan,False,1.0,146.680197,Indiana
625,875771,M,False,175200,Married,True,Homeowner with mortgage/loan,False,3.0,124.845087,Arkansas
667,931449,M,NaN,42400,Married,True,Homeowner free and clear,False,2.0,0.000000,Missouri
691,971703,M,False,-8700,Married,True,Homeowner with mortgage/loan,False,3.0,60.000000,Oklahoma
752,1047534,M,True,200000,Married,True,Homeowner with mortgage/loan,False,4.0,130.401605,Washington
868,1229417,F,True,48000,Married,True,Homeowner with mortgage/loan,False,3.0,0.000000,Washington
926,1303929,F,False,43400,Never Married,True,Rented,False,1.0,126.737284,West Virginia


In [6]:
df = df.drop([212, 266, 286, 593, 625, 667, 752, 868, 926, 953, 965])

df.loc[691, 'income'] = 8700

display(df.shape)

(933, 11)

In [7]:
df[['age', 'income', 'num.vehicles']].corr()

,age,income,num.vehicles
age,1.000000,-0.003035,-0.065713
income,-0.003035,1.000000,0.136994
num.vehicles,-0.065713,0.136994,1.000000


In comparing the raw and cleaned data, there is a clear difference in results. The correlation between age and income shifts from 0.027429 to -0.003035, a difference of 0.030464, which is a figure nearly as large as the raw data correlation. The correlation not only changes direction, but becomes significantly less powerful once the data is cleaned. There are not significant shifts observed in the raw and cleaned data correlation between age and number of vehicles, or income and number of vehicles. However, the strongest correlation observed is between income and number of vehicles, observed in both the raw and cleaned datasets. We can conclude that a significant portion of the shift in the age-income correlation is attributable to the extreme age outliers.

Cleaning Log

Fifty six entries removed with null values in columns 'income', 'num.vehicles', and 'age' (cell 3). Given that these are observational / reported data points, we cannot make inferences from the rest of the data set as to what they may contain, and replacing those data points with the median or mean would distort the data inward.

Eleven entries removed with impossible values for age or income (cell 4). Ages reported in the data either exceeded possible min / max (for example, the oldest living person in West Virginia is Virginia Elva Cadd who is listed as 107 years old as of September 2026). One entry listed with a negative income was changed to it's absolute value, as the entry is most likely attributable to a data entry error.